In [1]:
import iqpopt as iqp
import pennylane as qp
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
import random
from itertools import combinations
from pennylane.qnn import iqp_expval as op_expval

# IQP circuit

In [2]:
def number_partition_obs(numbers: list[int]) -> tuple[np.ndarray, np.ndarray]:
    """
    Construct the observables and coefficients corresponding to the
    Number Partitioning cost Hamiltonian.

    The Number Partitioning Hamiltonian is

        H = Σ_{i,j} numbers[i] * numbers[j] * O_{ij}

    where

        O_{ij} = I               if i == j
        O_{ij} = Z_i Z_j         if i != j

    In IQPopt, observables are represented as binary vectors:
    - 0 indicates the identity operator on a qubit.
    - 1 indicates a Pauli-Z operator on a qubit.

    For example, for 4 qubits:
        [1, 0, 1, 0]
    represents the observable Z_0 x Z_2.

    The identity observable is represented by:
        [0, 0, 0, 0]

    Parameters
    ----------
    numbers : list[int]
        List of integers defining the number partitioning instance.
        The length of the list determines the number of qubits.

    Returns
    -------
    ops : np.ndarray
        Array of shape (n_qubits^2, n_qubits) containing the observables.
        Each row is a binary vector encoding either an identity operator
        or a two-qubit Pauli-Z correlation.

    coeffs : np.ndarray
        Array of shape (n_qubits^2,) containing the coefficient associated
        with each observable. The k-th coefficient corresponds to the
        k-th observable in `ops`.
    """

    n_qubits = len(numbers)

    ops = []
    coeffs = []

    for i in range(n_qubits):
        for j in range(n_qubits):

            # Coefficient of O_{ij}
            coeffs.append(numbers[i] * numbers[j])

            # Binary representation of the observable
            ob = [0] * n_qubits

            # For i != j, encode Z_i Z_j
            if i != j:
                ob[i] = 1
                ob[j] = 1

            # For i == j, keep all zeros to represent Identity
            ops.append(ob)

    return np.array(ops), np.array(coeffs)

def gens(n_qubits: int, mode: str = "full") -> list:
    """
    Generate the list of IQP generator gates for an IQP circuit.

    Each generator is represented as a list containing a single term:
    - [[i]] represents a one-qubit Z rotation generator on qubit i.
    - [[i, j]] represents a two-qubit ZZ interaction generator between
      qubits i and j.

    The generated set always includes all single-qubit generators.
    Additional two-qubit generators are determined by the selected mode.

    Parameters
    ----------
    n_qubits : int
        Number of qubits in the IQP circuit.

    mode : str, optional
        Connectivity pattern for the two-qubit generators.

        Supported values are:

        - "full":
          Add a ZZ generator between every pair of qubits.
          This produces a fully connected interaction graph.

        - "circular":
          Add ZZ generators only between nearest neighbours in a ring.
          Qubit i is connected to i+1, and the last qubit is connected
          back to the first.

        Default is "full".

    Returns
    -------
    list
        List of IQP generators.

    Notes
    -----
    The total number of generators is:

    - mode="full":
      n_qubits + n_qubits * (n_qubits - 1) / 2

    - mode="circular":
      2 * n_qubits

    These generators can be passed directly to
    ``iqp.IqpSimulator(n_qubits, gates)``.
    """

    gates = []

    # Add all single-qubit generators.
    for i in range(n_qubits):
        gates.append([[i]])

    # Add all pairwise ZZ generators.
    if mode == "full":
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                gates.append([[i, j]])

    # Add nearest-neighbour ZZ generators on a ring.
    if mode == "circular":
        for i in range(n_qubits - 1):
            gates.append([[i, i + 1]])

        # Close the ring.
        gates.append([[n_qubits - 1, 0]])

    return gates

In [3]:
def loss_fn(
        params: jnp.ndarray,
        circuit,
        ops: np.ndarray,
        coeffs: np.ndarray,
        n_samples: int,
        key: jax.Array,
    ) -> jnp.ndarray:
    """
    Compute the expectation value of the Number Partitioning cost Hamiltonian.

    The cost function is evaluated as

        C(θ) = Σ_i coeffs[i] . ⟨O_i⟩

    where:
    - O_i are the observables specified in `ops`,
    - ⟨O_i⟩ are their expectation values under the IQP circuit
      parameterized by `params`,
    - coeffs[i] are the corresponding Hamiltonian coefficients.

    This function can be used directly with the IQPopt optimizer to
    variationally minimize the Number Partitioning objective.

    Parameters
    ----------
    params : jnp.ndarray
        Array of circuit parameters. The length must match the number
        of generators in the IQP circuit.

    circuit : iqp.IqpSimulator
        IQP circuit simulator containing the circuit structure and
        generator definitions.

    ops : np.ndarray
        Array of observables returned by `number_partition_obs`.
        Shape is (n_observables, n_qubits), where each row is a binary
        encoding of a Pauli-Z string.

    coeffs : np.ndarray
        Array of Hamiltonian coefficients associated with the
        observables in `ops`.
        Shape is (n_observables,).

    n_samples : int
        Number of Monte Carlo samples used to estimate expectation
        values.

    key : jax.Array
        JAX pseudo-random number generator key used for sampling.

    Returns
    -------
    jnp.ndarray
        Scalar expectation value of the Number Partitioning Hamiltonian.
        Lower values correspond to better partitions.

    Notes
    -----
    The expectation values are computed using IQPopt's fast estimator:

        expvals = op_expval(...)[0]

    and combined into the Hamiltonian expectation value through a
    weighted sum:

        C = coeffs . expvals

    which is equivalent to evaluating the expectation value of the
    corresponding PennyLane Hamiltonian.
    """

    # Estimate expectation values of all observables.
    expvals = op_expval(
        ops,
        params,
        circuit.gates,
        circuit.n_qubits,
        n_samples,
        key,
    )[0]

    # Compute the Hamiltonian expectation value.
    return jnp.dot(expvals, coeffs)

In [4]:
def train_model(
        circuit,
        ops: np.ndarray,
        coeffs: np.ndarray,
        params_init: np.ndarray,
        key: jax.Array,
        loss_fn,
        optimizer: str = "Adam",
        stepsize: float = 1e-3,
        n_iters: int = 4000,
        n_samples: int = 10000,
    ) -> iqp.Trainer:
    """
    Train an IQP circuit to minimize the cost function.

    Parameters
    ----------
    circuit : iqp.IqpSimulator
        IQP circuit simulator containing the circuit structure and
        generator definitions.

    ops : np.ndarray
        Array of observables defining the Hamiltonian.

    coeffs : np.ndarray
        Coefficients associated with the observables in `ops`.

    params_init : np.ndarray
        Initial parameters for the IQP circuit optimization.
        Must have length equal to `len(circuit.gates)`.

    key : jax.Array
        JAX pseudo-random number generator key used during expectation
        value estimation.

    loss_fn : callable
        Cost function used for optimization.

    optimizer : str, optional
        Name of the optimizer used by IQPopt.
        Default is "Adam".

    stepsize : float, optional
        Learning rate of the optimizer.
        Default is 0.001.

    n_iters : int, optional
        Number of optimization iterations.
        Default is 4000.

    n_samples : int, optional
        Number of samples used to estimate expectation values during
        training.
        Default is 10000.

    Returns
    -------
    iqp.Trainer
        Trained IQPopt trainer object containing the optimization
        results, including the final parameters in
        `trainer.final_params`.
    """

    trainer = iqp.Trainer(optimizer, loss_fn, stepsize)

    params_init = np.random.normal(0, 1, len(circuit.gates))

    loss_kwargs = {
        "params": params_init,
        "circuit": circuit,
        "ops": ops,
        "coeffs": coeffs,
        "n_samples": n_samples,
        "key": key,
    }

    trainer.train(n_iters, loss_kwargs)

    return trainer

In [5]:
# numbers =  [1, 3, 13, 7, 4]
# n_qubits = len(numbers)
# ops, coeffs = number_partition_obs(numbers)
# gates = gens(n_qubits, 'full')

# circuit = iqp.IqpSimulator(n_qubits, gates)

# # params = np.random.rand(len(gates))
# # n_samples = 1000
# # key = jax.random.PRNGKey(42)
# # expval, std = op_expval(ops, params, circuit.gates, circuit.n_qubits, n_samples, key)

# np.random.seed(0)

# # trainer = iqp.Trainer(optimizer, loss_fn, stepsize)
# params_init = np.random.normal(0, 1, len(circuit.gates))

# # trainer.train(n_iters, loss_kwargs, turbo=100) # the turbo option trains in iteration batches of the number that you input, using jit and lax.scan
# trainer =  train_model(circuit = circuit, 
#                        ops = ops, 
#                        coeffs = coeffs, 
#                        params_init =  params_init, 
#                        key = jax.random.PRNGKey(42), 
#                        loss_fn = loss_fn, 
#                        optimizer = "Adam", 
#                        stepsize = 1e-3, 
#                        n_iters = 4000, 
#                        n_samples = 1000)

# trained_params = trainer.final_params
# plt.plot(trainer.losses) # plot the loss curve
# plt.show()

In [6]:
# labels = [np.binary_repr(i, n_qubits) for i in range(2**n_qubits)]
# probs = circuit.probs(trained_params)
# # Plotting the histogram using matplotlib
# plt.figure(figsize=(8, 5))
# plt.bar(labels, probs, edgecolor='black', alpha=0.8)

# plt.xlabel('Basis State', fontsize=12)
# plt.ylabel('Probability', fontsize=12)
# plt.title('Quantum Circuit Measurement Probabilities', fontsize=14)
# plt.xticks(rotation=90)
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.tight_layout()
# plt.show()

# Problem generator

In [7]:
def count_partitions(nums):
    total = sum(nums)

    # Equal partition impossible if total is odd
    if total % 2 != 0:
        return 0

    target = total // 2
    n = len(nums)

    solutions = []

    for r in range(1, n):
        for subset in combinations(range(n), r):
            subset_sum = sum(nums[i] for i in subset)

            if subset_sum == target:
                other = tuple(sorted(set(range(n)) - set(subset)))

                # Canonical representation to avoid counting
                # A|B and B|A separately
                partition = tuple(sorted([
                    tuple(sorted(subset)),
                    other
                ]))

                if partition not in solutions:
                    solutions.append(partition)

    return len(solutions)

def generate_unique_partition_instance(
        n=8,
        min_value=1,
        max_value=30,
        max_attempts=100000):

    for _ in range(max_attempts):
        nums = [random.randint(min_value, max_value) for _ in range(n)]

        if count_partitions(nums) == 1:
            return nums

    return None

def sum_selected(binary_str, numbers):
    binary_arr = np.array(list(binary_str)) == '1'
    numbers_arr = np.array(numbers)

    if len(binary_arr) != len(numbers_arr):
        raise ValueError("Binary string and numbers list must have the same length")

    return numbers_arr[binary_arr].sum()

# Example usage
instance = generate_unique_partition_instance(6)

if instance:
    print("Instance with exactly one partition solution:")
    print(instance)
else:
    print("No instance found.")

Instance with exactly one partition solution:
[30, 23, 19, 18, 15, 29]


In [8]:
def can_partition(nums):
    total = sum(nums)

    # If total sum is odd, can't split equally
    if total % 2 != 0:
        return False

    target = total // 2

    # dp[i] = whether a subset sums to i
    dp = [False] * (target + 1)
    dp[0] = True

    for num in nums:
        for s in range(target, num - 1, -1):
            dp[s] = dp[s] or dp[s - num]

    return dp[target]

can_partition(instance)

True

# Benchmarking

In [9]:
dataset = {}
num_example = 100
max_num_of_numbers = 10

for i in range(3, max_num_of_numbers+1):
    data = []
    for _ in range(num_example):
        data.append(generate_unique_partition_instance(i))
    dataset[i] = data

In [10]:
exp = {}

In [ ]:
results = {}
for i in range(3, max_num_of_numbers+1):
    result = []
    for j in range(num_example):
        numbers = dataset[i][j]
        n_qubits = len(numbers)
        ops, coeffs = number_partition_obs(numbers)
        gates = gens(n_qubits, 'circular')
        circuit = iqp.IqpSimulator(n_qubits, gates)
        params_init = np.random.uniform(0, np.pi, len(circuit.gates))
        
        trainer =  train_model(circuit = circuit, 
                       ops = ops, 
                       coeffs = coeffs, 
                       params_init =  params_init, 
                       key = jax.random.PRNGKey(42), 
                       loss_fn = loss_fn, 
                       optimizer = "Adam", 
                       stepsize = 1e-3, 
                       n_iters = 2000, 
                       n_samples = 1000)

        trained_params = trainer.final_params
        
        probs = circuit.probs(trained_params)
        bits = np.binary_repr(np.argmax(probs), n_qubits)
        solution = sum_selected(bits, numbers)
        result.append(abs(solution - sum(numbers)//2) / (sum(numbers)//2))

    results[i] =  result

exp['circular'] = results

Training Progress: 100%|█| 2000/2000 [00:26<00:00, 75.45it/s, loss=0.000280, ela


Training has not converged after 2000 steps


Training Progress: 100%|█| 2000/2000 [00:12<00:00, 163.52it/s, loss=0.000010, el


Training has not converged after 2000 steps


Training Progress: 100%|█| 2000/2000 [00:12<00:00, 157.21it/s, loss=0.017346, el


...

Training Progress: 100%|█| 2000/2000 [00:11<00:00, 168.37it/s, loss=0.000358, el


Training has not converged after 2000 steps


Training Progress: 100%|█| 2000/2000 [00:11<00:00, 172.24it/s, loss=10.273486, e


Training has not converged after 2000 steps


Training Progress:  43%|▍| 857/2000 [00:05<00:07, 156.81it/s, loss=204.367528, e

In [ ]:
results = {}
for i in range(3, max_num_of_numbers+1):
    result = []
    for j in range(num_example):
        numbers = dataset[i][j]
        n_qubits = len(numbers)
        ops, coeffs = number_partition_obs(numbers)
        gates = gens(n_qubits, 'full')
        circuit = iqp.IqpSimulator(n_qubits, gates)
        params_init = np.random.uniform(0, np.pi, len(circuit.gates))
        
        trainer =  train_model(circuit = circuit, 
                       ops = ops, 
                       coeffs = coeffs, 
                       params_init =  params_init, 
                       key = jax.random.PRNGKey(42), 
                       loss_fn = loss_fn, 
                       optimizer = "Adam", 
                       stepsize = 1e-3, 
                       n_iters = 2000, 
                       n_samples = 1000)

        trained_params = trainer.final_params
        
        probs = circuit.probs(trained_params)
        bits = np.binary_repr(np.argmax(probs), n_qubits)
        solution = sum_selected(bits, numbers)
        result.append(abs(solution - sum(numbers)//2) / (sum(numbers)//2))

    results[i] =  result

exp['full'] = results

In [ ]:
results = {}
for i in range(3, max_num_of_numbers+1):
    result = []
    for j in range(num_example):
        numbers = dataset[i][j]
        n_qubits = len(numbers)
        ops, coeffs = number_partition_obs(numbers)
        gates = gens(n_qubits, 'single')
        circuit = iqp.IqpSimulator(n_qubits, gates)
        params_init = np.random.uniform(0, np.pi, len(circuit.gates))
        
        trainer =  train_model(circuit = circuit, 
                       ops = ops, 
                       coeffs = coeffs, 
                       params_init =  params_init, 
                       key = jax.random.PRNGKey(42), 
                       loss_fn = loss_fn, 
                       optimizer = "Adam", 
                       stepsize = 1e-3, 
                       n_iters = 2000, 
                       n_samples = 1000)

        trained_params = trainer.final_params
        
        probs = circuit.probs(trained_params)
        bits = np.binary_repr(np.argmax(probs), n_qubits)
        solution = sum_selected(bits, numbers)
        result.append(abs(solution - sum(numbers)//2) / (sum(numbers)//2))

    results[i] =  result

exp['single'] = results

In [ ]:
for i in ['circular', 'full', 'single']:
    for j in range(3, max_num_of_numbers+1):
        exp[i][j] = [np.mean(exp[i][j]), np.std(exp[i][j])]

In [ ]:
import pandas as pd

df = pd.DataFrame(exp)
df

In [ ]:
df.to_csv('output_random.csv')

In [1]:
import numpy as np
import random
from itertools import combinations

def count_partitions(nums: list[int]) -> int:
    """
    Count the number of unique ways to partition `nums` into two subsets
    with equal sum using brute-force enumeration.

    Each partition is counted only once (A|B is considered the same as B|A).

    Parameters
    ----------
    nums : list[int]
        List of integers to partition.

    Returns
    -------
    int
        Number of distinct equal-sum partitions.
        Returns 0 if total sum is odd.
    """

    total = sum(nums)

    # Equal partition impossible if total is odd
    if total % 2 != 0:
        return 0

    target = total // 2
    n = len(nums)

    solutions = []

    for r in range(1, n):
        for subset in combinations(range(n), r):
            subset_sum = sum(nums[i] for i in subset)

            if subset_sum == target:
                other = tuple(sorted(set(range(n)) - set(subset)))

                # Canonical representation to avoid counting
                # A|B and B|A separately
                partition = tuple(sorted([
                    tuple(sorted(subset)),
                    other
                ]))

                if partition not in solutions:
                    solutions.append(partition)

    return len(solutions)

def generate_unique_partition_instance(n: int = 8, 
                                       min_value: int = 1, 
                                       max_value: int = 30, 
                                       max_attempts: int = 100_000) -> list[int] | None:
    """
    Generate a number partition instance with exactly one valid equal-sum partition.

    The function randomly samples integer lists and keeps only those
    for which `count_partitions(nums) == 1`.

    Parameters
    ----------
    n : int
        Number of elements in the instance.

    min_value : int
        Minimum value of each integer.

    max_value : int
        Maximum value of each integer.

    max_attempts : int
        Maximum number of random samples to try before giving up.

    Returns
    -------
    list[int] | None
        A list of integers that admits exactly one equal-sum partition.
        Returns None if no such instance is found within `max_attempts`.

    Notes
    -----
    - This is a stochastic brute-force generator.
    - Runtime may be extremely slow for large `n`.
    """

    for _ in range(max_attempts):
        nums = [random.randint(min_value, max_value) for _ in range(n)]

        if count_partitions(nums) == 1:
            return nums

    return None

dataset = {}
num_example = 30
max_num_of_numbers = 10

for i in range(3, max_num_of_numbers+1):
    data = []
    count = 0
    while count < num_example:
        data.append(generate_unique_partition_instance(i))
        count += 1
    dataset[i] = data

In [3]:
import json

# Save dataset to disk
with open("partition_benchmark.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Dataset saved to partition_benchmark.json")

Dataset saved to partition_benchmark.json


In [5]:
import json

with open("partition_benchmark.json", "r") as f:
    dataset = json.load(f)

# Convert keys back to integers
dataset = {int(k): v for k, v in dataset.items()}

print("Dataset loaded successfully")

Dataset loaded successfully
